# Diagnose host source mapping drift and Atlas export cutoff


Bounded GT-HOST-TX-01 diagnostic only: compare original mapping hashes, primary export comments and a predeclared 0.5 per-region cutoff hypothesis from installed registry notes. Do not optimize a cutoff, replace data or conceal mismatches. Identify whether changed detection threshold explains the native replay discrepancy.

In [1]:
import pandas as pd, hashlib, gzip, json
from pathlib import Path
from starplast import host as H, organisms as O
root=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets')
base=Path('results/host_expression_source_review_2026_10_08')
sources={r['source_id']:r for r in json.loads((base/'sources.json').read_text())}
path=Path(next(f['absolute_path'] for f in sources['host_mouse_tissue_transcriptome']['verified_files'] if f['name']=='E-MTAB-3579-mouse-tissue-tpm.tsv'))
comments=[]
with path.open() as stream:
    for _ in range(10):
        line=stream.readline()
        if not line.startswith('#'):break
        comments.append(line.strip())
print('Source header comments',comments)
d=pd.read_csv(path,sep='\t',comment='#')
regions=list(H.FANTOM5_BRAIN)
print('Individual source minima',d[[*regions,H.FANTOM5_MUSCLE]].min().to_dict())
print('Mapping receipt comparisons')
hashes={}
for species,names in H.UNIPROT_IDMAP.items():
    for name in names:
        one=root/H.UNIPROT_ROOT/name
        other=root/'incoming_2026_09_25/uniprot'/name
        values={'current':hashlib.sha256(one.read_bytes()).hexdigest(),'earlier':hashlib.sha256(other.read_bytes()).hexdigest() if other.exists() else None}
        hashes[name]=values
        print(name,values)
(base/'mapping_hash_diagnostic.json').write_text(json.dumps(hashes,indent=2)+'\n')

Source header comments ['# Expression Atlas', "# Query: Genes matching: 'default query', specifically  expressed above the expression level cutoff: 0 tpms in experiment E-MTAB-3579", '# Selected columns: 170 (all)', '# Timestamp: Sat, 26-Sep-2026 23:23:19']
Individual source minima {'adult, cerebral cortex': 0.1, 'adult, cerebellum': 0.1, 'adult, hippocampal formation': 0.1, 'adult, olfactory brain': 0.2, 'juvenile, muscle (biceps femoris)': 0.1}
Mapping receipt comparisons
HUMAN_9606_idmapping.dat.gz {'current': '5fb356e81b07a938ac4d8e8a4b5e0faa89001bcff2aef59422ff30eaa8e72996', 'earlier': '5fb356e81b07a938ac4d8e8a4b5e0faa89001bcff2aef59422ff30eaa8e72996'}
reviewed_human.tsv.gz {'current': '842de4a8e8c961c0c6ad575579897ecc1b11425a6d851e71abfcf0ea50aa88fd', 'earlier': '842de4a8e8c961c0c6ad575579897ecc1b11425a6d851e71abfcf0ea50aa88fd'}
MOUSE_10090_idmapping.dat.gz {'current': 'c9cb0e15fb417b550ecd7151e3eed9f42eecb1e334cb35d174fca90fd292a2ea', 'earlier': 'c9cb0e15fb417b550ecd7151e3eed9f4

812

In [2]:
mapping=H.uniprot_index(str(root),'mouse')['by_ensembl']
d['host_id']=d['Gene ID'].str.split('.').str[0].map(mapping)
numeric=d[regions].apply(pd.to_numeric,errors='coerce')
selected=pd.DataFrame({'host_id':d.host_id,'brain_tpm':numeric.where(numeric>=.5).mean(axis=1),'skeletal_muscle_tpm':pd.to_numeric(d[H.FANTOM5_MUSCLE],errors='coerce').where(lambda x:x>=.5)})
selected=selected[selected.host_id.notna()].drop_duplicates('host_id').set_index('host_id')
installed=pd.read_parquet(Path('starplast/data')/O.HOST_TABLES[O.MOUSE]).set_index('host_id')
diagnostic={'hypothesis':'registry-documented 0.5 cutoff applied per original region before brain averaging; no threshold search','fields':{}}
for column in ('brain_tpm','skeletal_muscle_tpm'):
    candidate=selected[column].reindex(installed.index)
    known=installed[column].notna()
    measured=candidate.notna()
    overlap=known&measured
    diagnostic['fields'][column]={'missing':int((known&~measured).sum()),'extra':int((~known&measured).sum()),'numerical_differences_on_overlap':int(installed.loc[overlap,column].ne(candidate[overlap]).sum()),'tested_overlap':int(overlap.sum())}
(base/'cutoff_diagnostic.json').write_text(json.dumps(diagnostic,indent=2)+'\n')
print(diagnostic)

uniprot index (mouse): 15,656 Ensembl genes, 16,955 symbols; 55 Ensembl ids dropped for naming two reviewed proteins
{'hypothesis': 'registry-documented 0.5 cutoff applied per original region before brain averaging; no threshold search', 'fields': {'brain_tpm': {'missing': 8, 'extra': 22, 'numerical_differences_on_overlap': 0, 'tested_overlap': 9755}, 'skeletal_muscle_tpm': {'missing': 7, 'extra': 18, 'numerical_differences_on_overlap': 0, 'tested_overlap': 8628}}}


The diagnostic is not an admission/replacement and threshold-dependent non-detection remains unknown. The historical table and current public source can be numerically identical while the installed table used another detection filter; keep raw gene values separate from filtered legacy protein projections. Mapping hashes and exact source comments decide the next bounded transform review.